# LightRAG Evaluation — Embedding Model × Chunk Size Experiment

Compares two embedding models across multiple chunk sizes on multihop questions.

**Grid:** `len(EMBEDDING_MODELS) × len(CHUNK_SIZES)` experiments, each with its own isolated index.

**Runtime:** CPU is fine — all inference happens in OpenAI's cloud.

---
## Step 1 — Clone & install

In [ ]:
import os
if not os.path.exists('LightRAG'):
    !git clone https://github.com/anasmostafa23/LightRAG.git
os.chdir('LightRAG')
print('cwd:', os.getcwd())
!pip install -q -e ".[offline-llm]" nest_asyncio

---
## Step 2 — Configure the experiment

**Edit this cell — everything else runs automatically.**

In [ ]:
import os

# ── API Key ────────────────────────────────────────────────────────────────
OPENAI_API_KEY = 'sk-...'   # ← paste your key
# Or use Colab Secrets (🔑 sidebar → add OPENAI_API_KEY):
# from google.colab import userdata; OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

# ── Experiment grid ────────────────────────────────────────────────────────
# Two embedding models to compare.
# Each entry: (model_name, embedding_dimension)
# Dimensions: text-embedding-3-small → 1536, text-embedding-3-large → 3072
EMBEDDING_MODELS = [
    ('text-embedding-3-small', 1536),
    ('text-embedding-3-large', 3072),
]

# Chunk sizes to try (in tokens). LightRAG default is 1200.
CHUNK_SIZES = [512, 1024, 2048]

# ── LLM (for indexing, querying, and judging) ──────────────────────────────
LLM_MODEL   = 'gpt-4o-mini'   # indexing + querying
JUDGE_MODEL = 'gpt-4o-mini'   # scoring (swap to gpt-4o for better judgment)

# ── Query modes ────────────────────────────────────────────────────────────
MODES = ['naive', 'local', 'global', 'hybrid', 'mix']

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_WORKING_DIR = '/content/LightRAG/eval/rag_storage'
CORPUS_DIR       = '/content/LightRAG/eval/corpus'
QUESTIONS_FILE   = '/content/LightRAG/eval/questions.json'
RESULTS_DIR      = '/content/LightRAG/eval/results'

assert OPENAI_API_KEY.startswith('sk-'), 'Set your API key!'
print(f'Experiment grid: {len(EMBEDDING_MODELS)} models × {len(CHUNK_SIZES)} chunk sizes = {len(EMBEDDING_MODELS)*len(CHUNK_SIZES)} runs')
print(f'Models : {[m for m,_ in EMBEDDING_MODELS]}')
print(f'Chunks : {CHUNK_SIZES} tokens')

---
## Step 3 — Upload documents

Select your `.txt` or `.md` files. They are shared across all experiments.

In [ ]:
import os
from google.colab import files

os.makedirs(CORPUS_DIR, exist_ok=True)
print(f'Select your .txt / .md files → {CORPUS_DIR}')
uploaded = files.upload()

for fname, content in uploaded.items():
    dest = os.path.join(CORPUS_DIR, fname)
    with open(dest, 'wb') as f:
        f.write(content)
    print(f'  {dest}  ({len(content):,} bytes)')

docs = [f for f in os.listdir(CORPUS_DIR) if f.endswith(('.txt', '.md'))]
print(f'\nCorpus: {len(docs)} file(s) → {docs}')

---
## Step 4 — Define multihop questions

Fill in your real questions below. Required: `id`, `question`, `ground_truth`.
`hop_count` is stored in results for analysis — it is not used during querying.

In [ ]:
import json, os

questions = [
    {
        'id': 'q1',
        'hop_count': 2,
        'question': 'REPLACE: Your 2-hop question here.',
        'ground_truth': 'REPLACE: Full answer covering both hops.',
    },
    {
        'id': 'q2',
        'hop_count': 3,
        'question': 'REPLACE: Your 3-hop question here.',
        'ground_truth': 'REPLACE: Full answer covering all three hops.',
    },
]

os.makedirs(os.path.dirname(QUESTIONS_FILE), exist_ok=True)
with open(QUESTIONS_FILE, 'w') as f:
    json.dump({'questions': questions}, f, indent=2)
print(f'{len(questions)} question(s) saved to {QUESTIONS_FILE}')

---
## Step 5 — Run all experiments

Iterates over every (embedding model, chunk size) combination in the grid.
Each run builds its own isolated index under `eval/rag_storage/<tag>/`.
Results are saved per-experiment and also merged into one comparison CSV.

In [ ]:
import sys, asyncio, copy, nest_asyncio
nest_asyncio.apply()

REPO_ROOT = '/content/LightRAG'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from eval.config import EvalConfig
from eval.pipeline import run

all_scores = []   # collect Score objects from all runs for comparison

for embed_model, embed_dim in EMBEDDING_MODELS:
    for chunk_size in CHUNK_SIZES:
        print(f'\n{"="*60}')
        print(f'  model={embed_model}  chunk={chunk_size}')
        print(f'{"="*60}')

        cfg = EvalConfig(
            backend            = 'openai',
            openai_api_key     = OPENAI_API_KEY,
            openai_llm_model   = LLM_MODEL,
            openai_embed_model = embed_model,
            openai_embed_dim   = embed_dim,
            judge_model        = JUDGE_MODEL,
            chunk_token_size        = chunk_size,
            chunk_overlap_token_size= max(50, chunk_size // 10),  # 10% overlap
            modes              = MODES,
            working_dir        = BASE_WORKING_DIR,
            corpus_dir         = CORPUS_DIR,
            questions_file     = QUESTIONS_FILE,
            results_dir        = RESULTS_DIR,
            skip_ingestion     = False,  # always build fresh per config
        )

        scores = asyncio.run(run(cfg))
        all_scores.extend(scores)

print('\nAll experiments complete.')

---
## Step 6 — Compare results across experiments

In [ ]:
import pandas as pd, glob, os

# Load all per-experiment CSVs and merge
csv_files = sorted(glob.glob(os.path.join(RESULTS_DIR, '*.csv')))
if not csv_files:
    print('No results yet — run Step 5 first.')
else:
    df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

    score_cols = ['faithfulness', 'answer_relevance', 'completeness', 'coherence', 'overall']

    print('=== Mean scores by experiment (embed model + chunk size) ===')
    by_exp = df.groupby('experiment')[score_cols].mean().round(3)
    display(by_exp.sort_values('overall', ascending=False))

    print('\n=== Mean scores by embed model (averaged over all chunk sizes) ===')
    # Extract model name from experiment tag (everything before __chunk)
    df['embed_model'] = df['experiment'].str.replace(r'__chunk\d+', '', regex=True)
    display(df.groupby('embed_model')[score_cols].mean().round(3))

    print('\n=== Mean scores by chunk size (averaged over both models) ===')
    df['chunk_size'] = df['experiment'].str.extract(r'chunk(\d+)').astype(int)
    display(df.groupby('chunk_size')[score_cols].mean().round(3))

    print('\n=== Mean scores by RAG mode (averaged over all experiments) ===')
    display(df.groupby('mode')[score_cols].mean().round(3))

In [ ]:
# Optional: heatmap of overall score — experiment × mode
import matplotlib.pyplot as plt
import seaborn as sns

pivot = df.pivot_table(values='overall', index='experiment', columns='mode', aggfunc='mean').round(2)
fig, ax = plt.subplots(figsize=(10, max(4, len(pivot) * 0.6)))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', vmin=1, vmax=5,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Overall score (1–5)'})
ax.set_title('LightRAG — Overall score by experiment × mode')
ax.set_ylabel('Experiment (embed model + chunk size)')
ax.set_xlabel('Query mode')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'heatmap.png'), dpi=150)
plt.show()
print('Saved heatmap.png')

In [ ]:
# Save the merged comparison CSV and download everything
from google.colab import files as colab_files

merged_path = os.path.join(RESULTS_DIR, 'comparison_all.csv')
df.to_csv(merged_path, index=False)
print(f'Saved merged CSV → {merged_path}')

# Download all result files
for path in sorted(glob.glob(os.path.join(RESULTS_DIR, '*'))):
    if os.path.isfile(path):
        print(f'Downloading {os.path.basename(path)}')
        colab_files.download(path)

---
## Notes

### What each experiment tag means
```
text-embedding-3-small__chunk512
text-embedding-3-small__chunk1024
text-embedding-3-small__chunk2048
text-embedding-3-large__chunk512
text-embedding-3-large__chunk1024
text-embedding-3-large__chunk2048
```
Each has its own isolated index under `eval/rag_storage/<tag>/`.

### Re-running without re-indexing
If you already built the indexes and just want to re-run queries or change questions:
```python
cfg.skip_ingestion = True
```

### Chunk overlap
Overlap is set to 10% of chunk size automatically (`max(50, chunk_size // 10)`).
Change the formula in Step 5 if you want fixed overlap.

### Cost estimate (rough)
6 experiments × (indexing + 5 modes × N questions + judging):
- Indexing dominates: ~$0.01–$0.10 per document per run with `gpt-4o-mini`
- `text-embedding-3-large` costs ~6× more than `small` for embeddings
- Total for a small corpus (5 docs) + 5 questions: roughly **$0.50–$2.00**

### Scoring dimensions
| Metric | Measures |
|---|---|
| faithfulness | No hallucinations |
| answer_relevance | Directly addresses the question |
| completeness | All multihop reasoning steps covered |
| coherence | Logically structured answer |
| overall | Mean of the four above |